# Hotel Data Cleaning

This notebook demonstrates the initial cleaning and preparation workflow for the hotel dataset used in the accessibility analysis of my dog-friendly travel project. The dataset covers five German target cities (Berlin, Hamburg, Leipzig, München, and Hannover) and is sourced from OpenStreetMap (OSM) GeoJSON extractions.

This is an example notebook illustrating the data preparation process, not the final dataset or complete analysis. The raw and processed datasets are stored locally and are not included in the GitHub repository. Therefore, the notebook cannot be run without obtaining the source files and placing them in the expected local directories.

The goal is to create a consistent dataset of accommodation locations that can later be combined with geographic data on veterinary practices, parks, and pet shops.

The cleaning process includes data inspection, column selection, missing-value checks, duplicate checks, and data validation.


# 1. Initial data exploration 

Data exploration with one file (Hamburg) to understand the structure and make decision on data cleaning. 

In [1]:
import pandas as pd

# Load test file
file_path = "../data/raw/hamburg_hotels.geojson"
df = pd.read_json(file_path)

# 2. Flatten the 'features' column
# This automatically unpacks 'properties', 'geometry', and 'id' into columns
flat_df = pd.json_normalize(df["features"])

# 3. Check the resulting columns
print(f"New shape: {flat_df.shape}")
print(flat_df.columns.tolist())
display(flat_df.head(2))

New shape: (62258, 324)
['type', 'id', 'properties.@id', 'properties.building', 'properties.building:levels', 'properties.roof:levels', 'properties.roof:shape', 'properties.type', 'geometry.type', 'geometry.coordinates', 'properties.building:material', 'properties.roof:colour', 'properties.dsa:criteria', 'properties.heritage', 'properties.heritage:operator', 'properties.historic', 'properties.image', 'properties.ref:dsa', 'properties.start_date', 'properties.amenity', 'properties.contact:website', 'properties.description', 'properties.name', 'properties.operator', 'properties.social_facility', 'properties.social_facility:for', 'properties.addr:city', 'properties.addr:country', 'properties.addr:housenumber', 'properties.addr:postcode', 'properties.addr:street', 'properties.brand', 'properties.brand:wikidata', 'properties.check_date', 'properties.contact:email', 'properties.contact:phone', 'properties.guest_house', 'properties.internet_access', 'properties.internet_access:fee', 'properti

,type,id,properties.@id,properties.building,properties.building:levels,properties.roof:levels,properties.roof:shape,properties.type,geometry.type,geometry.coordinates,...,properties.check_in,properties.check_out,properties.payment:coins,properties.payment:notes,properties.name:ru,properties.capacity:rooms,properties.gym,properties.mobile,properties.dryer,properties.washing_machine
0,Feature,relation/39126,relation/39126,apartments,5,0,flat,multipolygon,Polygon,"[[[10.0776955, 53.5521073], [10.0777777, 53.55...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Feature,relation/49148,relation/49148,apartments,5,NaN,flat,multipolygon,Polygon,"[[[10.0607784, 53.5630334], [10.0607043, 53.56...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# 1. Strip 'properties.' from the column names to make them readable
flat_df.columns = flat_df.columns.str.replace("properties.", "", regex=False)

# 2. Check which columns actually have the most data (non-null counts)
# This shows us the top 20 most frequently used columns in this file
print("Top 20 most populated columns:")
print(flat_df.notnull().sum().sort_values(ascending=False).head(20))

Top 20 most populated columns:
type                    62258
id                      62258
@id                     62258
geometry.type           62258
geometry.coordinates    62258
building                61947
addr:street             50646
addr:housenumber        50438
building:levels         50303
addr:postcode           47438
addr:city               47256
roof:shape              41041
roof:levels             40755
addr:country            38367
building:material        3871
roof:material            1913
roof:orientation         1899
roof:colour              1748
ref:dsa                  1243
heritage                 1232
dtype: int64


In [3]:
# View all columns with their non-null count, sorted from most filled to least filled
column_counts = flat_df.notnull().sum().sort_values(ascending=False)
for col, count in column_counts.items():
  print(f"{col}: {count} rows")

type: 62258 rows
id: 62258 rows
@id: 62258 rows
geometry.type: 62258 rows
geometry.coordinates: 62258 rows
building: 61947 rows
addr:street: 50646 rows
addr:housenumber: 50438 rows
building:levels: 50303 rows
addr:postcode: 47438 rows
addr:city: 47256 rows
roof:shape: 41041 rows
roof:levels: 40755 rows
addr:country: 38367 rows
building:material: 3871 rows
roof:material: 1913 rows
roof:orientation: 1899 rows
roof:colour: 1748 rows
ref:dsa: 1243 rows
heritage: 1232 rows
heritage:operator: 1231 rows
start_date: 1199 rows
building:colour: 1026 rows
source: 961 rows
name: 840 rows
height: 658 rows
addr:suburb: 560 rows
tourism: 531 rows
roof:height: 503 rows
dsa:criteria: 438 rows
owner: 431 rows
wheelchair: 412 rows
check_date: 374 rows
construction_year: 373 rows
architect: 357 rows
website: 351 rows
operator: 329 rows
phone: 257 rows
construction_date: 239 rows
internet_access: 225 rows
building:flats: 217 rows
historic: 214 rows
description: 173 rows
contact:website: 172 rows
contact:ph

### CLASSIFY ACCOMODATION RECORDS 

Identify tourism-tagged locations and buildings


In [ ]:
# 1. Identify locations with an explicit tourism tag
tourism_mask = flat_df["tourism"].notna()
clean_df = flat_df[tourism_mask].copy()

# 2. Identify buildings without a tourism tag
building_mask = flat_df["tourism"].isna() & flat_df["building"].notna()
building_only = flat_df[building_mask].copy()

print(f"Tourism-tagged locations: {len(clean_df)}")
print(f"Building-only locations: {len(building_only)}")


Tourism-tagged locations: 531
Building-only locations: 61727


### IDENTIFY POTENTIAL MISSING ACCOMMODATIONS

Use names and pet-related tags to find additional candidates.

In [ ]:
# Check for accommodation-related keywords in names
keywords = [
    "ferienwohnung",
    "apartment",
    "hotel",
    "pension",
    "gästehaus",
    "hostel"
]

if "name" in building_only.columns:
    mask_named = building_only["name"].str.contains(
        "|".join(keywords),
        case=False,
        na=False
    )
else:
    mask_named = pd.Series(False, index=building_only.index)

# Check for explicit pet-related tags
pet_columns = [
    col for col in ["dog", "pets_allowed"]
    if col in building_only.columns
]

if pet_columns:
    mask_pets = building_only[pet_columns].notna().any(axis=1)
else:
    mask_pets = pd.Series(False, index=building_only.index)

# Keep buildings matching either condition
rescued_buildings = building_only[mask_named | mask_pets].copy()

print(f"Potential accommodations identified in buildings: {len(rescued_buildings)}")

Potential accommodations identified in buildings: 14


Important: These are potential accommodations, not necessarily confirmed ones. A building name containing "hotel" or a pet-related tag does not automatically establish that the feature is a bookable accommodation.

### COMBINE AND DEDUPLICATE 

Create the resulting dataset and inspect it.

In [ ]:
# Inspect locations with explicit dog-related tags
if "dog" in flat_df.columns:
    dogs_explicit = flat_df[flat_df["dog"].notna()]

    columns_to_show = [
        col for col in ["name", "tourism", "dog", "website", "addr:city"]
        if col in dogs_explicit.columns
    ]

    display(dogs_explicit[columns_to_show])

# Combine tourism-tagged locations with potential accommodations
final_df = pd.concat(
    [clean_df, rescued_buildings],
    ignore_index=True
)

# Remove duplicate OSM elements if an ID is available
if "@id" in final_df.columns:
    final_df = final_df.drop_duplicates(subset="@id")
else:
    final_df = final_df.drop_duplicates()

print(f"Final dataset size: {len(final_df)} locations")

,name,tourism,dog,website,addr:city
119,Campingplatz Schnelsen-Nord,camp_site,yes,https://www.knauscamp.de/hamburg,Hamburg
130,Wohnmobilhafen Hamburg Süd,caravan_site,yes,https://www.hamburg-tourism.de/sehen-erleben/g...,NaN
1774,Motel One Hamburg-Altona,hotel,yes,NaN,Hamburg
3069,Motel One Hamburg-Alster,hotel,yes,https://www.motel-one.com/de/hotels/hamburg/ho...,Hamburg
17233,Leonardo Hotel Hamburg-Stillhorn,hotel,yes,NaN,Hamburg
23977,Motel One Hamburg-Airport,hotel,yes,https://www.motel-one.com/de/hotels/hamburg/ho...,Hamburg
41434,Prize by Radisson Hamburg City,hotel,no,NaN,Hamburg
61670,Servicewohnen Rahlstedt Deichgrafenhaus,NaN,leashed,NaN,Hamburg
62072,Motel One Hamburg am Michel,hotel,yes,https://www.motel-one.com/de/hotels/hamburg/ho...,Hamburg
62154,Motel One Hamburg-Fleetinsel,hotel,yes,NaN,Hamburg


Final dataset size: 545 locations


In [8]:
# View all columns again with their non-null count, sorted from most filled to least filled
column_counts_final = final_df.notnull().sum().sort_values(ascending=False)
for col, count in column_counts_final.items():
  print(f"{col}: {count} rows")

type: 545 rows
@id: 545 rows
id: 545 rows
geometry.type: 545 rows
geometry.coordinates: 545 rows
name: 536 rows
tourism: 531 rows
addr:housenumber: 429 rows
addr:street: 425 rows
addr:postcode: 405 rows
addr:city: 400 rows
wheelchair: 298 rows
website: 279 rows
check_date: 260 rows
addr:country: 258 rows
building: 234 rows
internet_access: 222 rows
phone: 222 rows
email: 146 rows
contact:website: 143 rows
contact:phone: 141 rows
addr:suburb: 139 rows
building:levels: 139 rows
internet_access:fee: 114 rows
contact:email: 111 rows
operator: 103 rows
brand: 98 rows
stars: 96 rows
wikidata: 92 rows
source: 90 rows
roof:shape: 86 rows
roof:levels: 85 rows
brand:wikidata: 85 rows
rooms: 82 rows
contact:fax: 77 rows
fax: 65 rows
toilets:wheelchair: 59 rows
smoking: 50 rows
air_conditioning: 46 rows
payment:cash: 28 rows
start_date: 24 rows
opening_hours: 24 rows
bar: 22 rows
brand:wikipedia: 22 rows
contact:facebook: 16 rows
fee: 15 rows
payment:credit_cards: 14 rows
heritage: 13 rows
alt_nam

### DROP EMPTY COLUMNS

Columns where all values are NaN are dropped. 

In [9]:
final_df = final_df.dropna(axis=1, how='all').copy()

print(f"Columns remaining: {final_df.shape[1]}")
print(f"Total rows: {final_df.shape[0]}")

Columns remaining: 200
Total rows: 545


### IDENTIFY THE COLUMNS TO KEEP 

Reduce the 200 columns to those that are relevant for the analysis. Kepp important imformation like classification, address info, contact info, online presence, accessibility, spatial data, dog information, payment information and special amenities. 

In [10]:
columns_to_keep = [
    # 1. Core Identity & Classification
    "type",
    "id",
    "@id",
    "name",
    "tourism",
    "building",
    "brand",
    "operator",
    "stars",
    "rooms",
    "guest_house",
    "cabins", 
    "caravan_site", 
    "caravan_site",
     # 2. Address Info
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "addr:country",
    "addr:suburb",
    # 3. Contact Info
    "phone",
    "contact:phone",
    "email",
    "contact:email",
    "fax",
    "contact:fax",
    # 4. Online Presence
    "website",
    "contact:website",
    "wikidata",
    "wikipedia",
    "brand:wikidata",
    "contact:facebook",
    "contact:instagram",
    "source",
    # 5. Accessibility
    "wheelchair",
    "toilets:wheelchair",
    # 6. Spatial Data
    "geometry.type",
    "geometry.coordinates",
    # 7. Pets / Dogs
    "dog",
    "pets_allowed",
    # 8. Payments & Financial Info (What OSM uses instead of prices)
    "payment:cash",
    "payment:credit_cards",
    "payment:debit_cards",
    "payment:visa",
    "payment:mastercard",
    "payment:contactless",
    # 9. Useful Hotel Amenities (Good additions!)
    "breakfast",
    "internet_access",
    "internet_access:fee",
    "air_conditioning",
    "smoking",
    "bar",
    "swimming_pool",
    "sauna",
    "gym",
    "opening_hours",
]





In [11]:
# Keep only the columns from our list that actually exist in this specific file
# (Using intersection prevents errors if a file is missing one of the columns)
cols_present = [col for col in columns_to_keep if col in final_df.columns]
slim_df = final_df[cols_present].copy()

print(f"Old shape: {flat_df.shape} --> New slim shape: {slim_df.shape}")

Old shape: (62258, 324) --> New slim shape: (545, 57)


### COMBINE OVERLAPPING FIELDS 

Some columns have overlapping information - this information is collected in one single column. The Rest is dropped.

In [12]:
# 1. Combine overlapping fields (take primary if available, fallback to contact:)
if "phone" in slim_df.columns and "contact:phone" in slim_df.columns:
  slim_df["phone"] = slim_df["phone"].fillna(slim_df["contact:phone"])

if "website" in slim_df.columns:
  if "contact:website" in slim_df.columns:
    slim_df["website"] = slim_df["website"].fillna(slim_df["contact:website"])
  if "source" in slim_df.columns:
    slim_df["website"] = slim_df["website"].fillna(slim_df["source"])

if "email" in slim_df.columns and "contact:email" in slim_df.columns:
  slim_df["email"] = slim_df["email"].fillna(slim_df["contact:email"])

if "fax" in slim_df.columns and "contact:fax" in slim_df.columns:
  slim_df["fax"] = slim_df["fax"].fillna(slim_df["contact:fax"])
  
  

# 2. Combine dog and pets_allowed into one clean column
if "dog" in slim_df.columns and "pets_allowed" in slim_df.columns:
  slim_df["pets_allowed_clean"] = slim_df["dog"].fillna(
      slim_df["pets_allowed"]
  )
  
  

# 3. Rename columns to make them clean and drop the 'contact:' prefix where desired
rename_map = {
    "addr:street": "street",
    "addr:housenumber": "housenumber",
    "addr:postcode": "postcode",
    "addr:city": "city",
    "addr:country": "country",
    "addr:suburb": "suburb",
    "geometry.type": "geom_type",
    "geometry.coordinates": "coordinates",
    "contact:facebook": "facebook",
    "contact:instagram": "instagram",
    "contact:twitter": "twitter",
}

slim_df = slim_df.rename(columns=rename_map)


# 4. Drop the old redundant contact columns we just merged
cols_to_drop_now = [
    "contact:phone",
    "contact:website",
    "contact:email",
    "contact:fax",
    "dog",
    "source",
]
slim_df = slim_df.drop(
    columns=[c for c in cols_to_drop_now if c in slim_df.columns]
)

print(f"Cleaned columns count: {len(slim_df.columns)}")
print(slim_df.columns.tolist())

Cleaned columns count: 52
['type', 'type', 'id', '@id', 'name', 'tourism', 'building', 'brand', 'operator', 'stars', 'rooms', 'guest_house', 'cabins', 'caravan_site', 'caravan_site', 'street', 'housenumber', 'postcode', 'city', 'country', 'suburb', 'phone', 'email', 'fax', 'website', 'wikidata', 'wikipedia', 'brand:wikidata', 'facebook', 'instagram', 'wheelchair', 'toilets:wheelchair', 'geom_type', 'coordinates', 'pets_allowed', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'payment:visa', 'payment:mastercard', 'payment:contactless', 'breakfast', 'internet_access', 'internet_access:fee', 'air_conditioning', 'smoking', 'bar', 'swimming_pool', 'sauna', 'gym', 'opening_hours', 'pets_allowed_clean']


In [13]:
# If there are duplicate column names, keep only unique ones or drop the extra 'type'
slim_df = slim_df.loc[:, ~slim_df.columns.duplicated()]

# 2. Batch Data Cleaning and Merging

This function automates the cleaning and standardization of multiple OpenStreetMap GeoJSON files containing accommodation data.

First, each file is loaded and its geographic features are converted into a structured DataFrame. The city name is inferred from the filename and used as a fallback when address information is missing.

To retain relevant accommodation records, the function selects locations with a `tourism` tag and potential additional accommodations identified through building tags, accommodation-related keywords, or explicit pet-related tags. This helps identify possible accommodation records that may not be classified under the `tourism` key in OpenStreetMap.

Next, the function selects relevant columns covering accommodation details, addresses, contact information, amenities, accessibility, pet-related tags, and geographic data. Where possible, overlapping contact fields are consolidated, missing city names are filled, and latitude and longitude are extracted from the geographic coordinates. Finally, columns are renamed for consistency, and redundant fields are removed.

The function is then applied to all matching hotel GeoJSON files in the raw data folder. Successfully cleaned DataFrames are combined into a single master DataFrame, creating a standardized dataset for further data quality checks and geographic accessibility analysis.

**Note:** The selection criteria identify potential accommodations, but do not guarantee that every retained record represents a commercial or dog-friendly accommodation. Further validation and duplicate checks may be required before the final analysis.



In [15]:
import glob
import os
import json
import pandas as pd


def clean_geojson_file(file_path):
  # 1. Extract category/city name from filename
  file_name = os.path.basename(file_path)
  inferred_city = file_name.split(".")[0].replace("_", " ").title()

  # 2. Load the file & flatten the 'features' list
  try:
    with open(file_path, "r", encoding="utf-8") as f:
      data = json.load(f)
  except Exception:
    return pd.DataFrame()

  features = data.get("features", [])
  if not features:
    return pd.DataFrame()

  flat_df = pd.json_normalize(features)
  flat_df.columns = flat_df.columns.str.replace("properties.", "", regex=False)

  # --- ROW FILTERING: Keep only commercial accommodations & rescued buildings ---
  is_tourism = (
      flat_df["tourism"].notna() if "tourism" in flat_df.columns else pd.Series(False, index=flat_df.index)
  )
  
  keywords = ["ferienwohnung", "apartment", "hotel", "pension", "gästehaus", "hostel"]
  pattern = "|".join(keywords)
  
  has_name = flat_df["name"].notna() if "name" in flat_df.columns else pd.Series(False, index=flat_df.index)
  has_building = flat_df["building"].notna() if "building" in flat_df.columns else pd.Series(False, index=flat_df.index)
  has_commercial_name = has_name & flat_df["name"].astype(str).str.contains(pattern, case=False, na=False)
  
  has_pet_tag = pd.Series(False, index=flat_df.index)
  if "dog" in flat_df.columns:
    has_pet_tag = has_pet_tag | flat_df["dog"].notna()
  if "pets_allowed" in flat_df.columns:
    has_pet_tag = has_pet_tag | flat_df["pets_allowed"].notna()

  is_rescued = has_building & (has_commercial_name | has_pet_tag)
  
  # Apply filter
  flat_df = flat_df[is_tourism | is_rescued].copy()
  if flat_df.empty:
    return pd.DataFrame()
  # ---------------------------------------------------------------------------

  # 3. Define the columns we want to keep (Organized & Expanded)
  columns_to_keep = [
      # 1. Core Identity & Classification
      "type",
      "id",
      "@id",
      "name",
      "tourism",
      "building",
      "brand",
      "operator",
      "stars",
      "rooms",
      "guest_house",
      "cabins",
      "caravan_site",
      
      # 2. Address Info
      "addr:street",
      "addr:housenumber",
      "addr:postcode",
      "addr:city",
      "addr:country",
      "addr:suburb",
      
      # 3. Contact Info
      "phone",
      "contact:phone",
      "email",
      "contact:email",
      "fax",
      "contact:fax",
      
      # 4. Online Presence
      "website",
      "contact:website",
      "wikidata",
      "wikipedia",
      "brand:wikidata",
      "contact:facebook",
      "contact:instagram",
      "source",
      
      # 5. Accessibility
      "wheelchair",
      "toilets:wheelchair",
      
      # 6. Spatial Data
      "geometry.type",
      "geometry.coordinates",
      
      # 7. Pets / Dogs
      "dog",
      "pets_allowed",
      
      # 8. Payments & Financial Info
      "payment:cash",
      "payment:credit_cards",
      "payment:debit_cards",
      "payment:visa",
      "payment:mastercard",
      "payment:contactless",
      
      # 9. Useful Hotel Amenities
      "breakfast",
      "internet_access",
      "internet_access:fee",
      "air_conditioning",
      "smoking",
      "bar",
      "swimming_pool",
      "sauna",
      "gym",
      "opening_hours",
  ]

  cols_present = [col for col in columns_to_keep if col in flat_df.columns]
  slim_df = flat_df[cols_present].copy()

  # 4. Merge overlapping contact info safely (including source fallback for website)
  if "phone" in slim_df.columns and "contact:phone" in slim_df.columns:
    slim_df["phone"] = slim_df["phone"].fillna(slim_df["contact:phone"])
    
  if "website" in slim_df.columns:
    if "contact:website" in slim_df.columns:
      slim_df["website"] = slim_df["website"].fillna(slim_df["contact:website"])
    if "source" in slim_df.columns:
      slim_df["website"] = slim_df["website"].fillna(slim_df["source"])
      
  if "email" in slim_df.columns and "contact:email" in slim_df.columns:
    slim_df["email"] = slim_df["email"].fillna(slim_df["contact:email"])
  if "fax" in slim_df.columns and "contact:fax" in slim_df.columns:
    slim_df["fax"] = slim_df["fax"].fillna(slim_df["contact:fax"])

  # --- ROBUST PET TAG COMBINATION ---
  if "dog" not in slim_df.columns:
    slim_df["dog"] = None
  if "pets_allowed" not in slim_df.columns:
    slim_df["pets_allowed"] = None

  slim_df["pets_allowed_clean"] = slim_df["dog"].fillna(slim_df["pets_allowed"])
  # ----------------------------------

  # Insert inferred city if 'addr:city' is missing/blank
  if "addr:city" in slim_df.columns:
    slim_df["addr:city"] = slim_df["addr:city"].fillna(inferred_city)
  else:
    slim_df["addr:city"] = inferred_city

  # 5. Extract Latitude and Longitude from coordinates
  def extract_lat_lon(coord):
    if not isinstance(coord, list) or len(coord) == 0:
      return None, None
    if len(coord) == 2 and isinstance(coord[0], (int, float)):
      return coord[1], coord[0]
    elif isinstance(coord[0], list):
      try:
        return coord[0][0][1], coord[0][0][0]
      except (IndexError, TypeError):
        return None, None
    return None, None

  if "geometry.coordinates" in slim_df.columns:
    lat_lons = slim_df["geometry.coordinates"].apply(extract_lat_lon)
    slim_df["latitude"] = [x[0] for x in lat_lons]
    slim_df["longitude"] = [x[1] for x in lat_lons]

  # 6. Rename columns & drop raw/redundant ones
  rename_map = {
      "addr:street": "street",
      "addr:housenumber": "housenumber",
      "addr:postcode": "postcode",
      "addr:city": "city",
      "addr:country": "country",
      "addr:suburb": "suburb",
      "geometry.type": "geom_type",
      "geometry.coordinates": "coordinates",
      "contact:facebook": "facebook",
      "contact:instagram": "instagram",
  }
  slim_df = slim_df.rename(columns=rename_map)

  cols_to_drop = [
      "contact:phone",
      "contact:website",
      "contact:email",
      "contact:fax",
      "dog",
      "pets_allowed",
      "source",  # Clean up source column since its useful data is now folded into website
  ]
  slim_df = slim_df.drop(columns=[c for c in cols_to_drop if c in slim_df.columns])
  slim_df = slim_df.loc[:, ~slim_df.columns.duplicated()]

  return slim_df


# --- BATCH RUN ACROSS FILES ---
folder_path = "../data/raw/*hotel*.geojson"
file_list = glob.glob(folder_path)

print(f"Found {len(file_list)} hotel files.")

all_dfs = []
for f in file_list:
  cleaned_df = clean_geojson_file(f)
  if not cleaned_df.empty:
    all_dfs.append(cleaned_df)

# Merge everything into the master dataframe
master_df_hotels = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

print(
    f"Successfully processed {len(file_list)} files! "
    f"Master DataFrame shape: {master_df_hotels.shape}"
)
if not master_df_hotels.empty:
  display(master_df_hotels.head())

Found 5 hotel files.
Successfully processed 5 files! Master DataFrame shape: (2517, 51)


,type,id,@id,name,tourism,building,brand,operator,stars,rooms,...,air_conditioning,smoking,bar,swimming_pool,sauna,gym,opening_hours,pets_allowed_clean,latitude,longitude
0,Feature,relation/446134,relation/446134,A&O Hostel Hamburg City,hostel,industrial,A&O,A&O,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,24/7,NaN,53.551009,10.023183
1,Feature,relation/2294560,relation/2294560,Bettenburg Hotel & Hostel,hotel,yes,NaN,evolvem Verwaltung GmbH,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.488233,10.186891
2,Feature,relation/3447704,relation/3447704,Vier Jahreszeiten,hotel,yes,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.555517,9.990784
3,Feature,relation/7922459,relation/7922459,Fraser Suites Hamburg,hotel,yes,NaN,Frasers Hospitality Group,5,154,...,NaN,outside,NaN,NaN,NaN,NaN,NaN,NaN,53.548772,9.986202
4,Feature,relation/9075174,relation/9075174,Camping Oortkaten,camp_site,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Apr-Oct,NaN,"[10.1103148, 53.4400379]","[10.1100935, 53.4399884]"


# 3. Data Quality Assessment

The cleaned dataset is assessed for data quality and consistency. This includes inspecting data types, identifying missing values and duplicate records, and checking the completeness of key attributes. Geographic coordinates are also reviewed to identify missing or invalid locations before conducting the spatial accessibility analysis.

### INSPECTING DATA TYPES 

In [16]:
master_df_hotels.info()

<class 'pandas.DataFrame'>
RangeIndex: 2517 entries, 0 to 2516
Data columns (total 51 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   type                  2517 non-null   str   
 1   id                    2517 non-null   str   
 2   @id                   2517 non-null   str   
 3   name                  2426 non-null   str   
 4   tourism               2477 non-null   str   
 5   building              776 non-null    str   
 6   brand                 427 non-null    str   
 7   operator              483 non-null    str   
 8   stars                 385 non-null    str   
 9   rooms                 293 non-null    str   
 10  guest_house           27 non-null     str   
 11  cabins                9 non-null      str   
 12  caravan_site          10 non-null     str   
 13  street                1954 non-null   str   
 14  housenumber           1939 non-null   str   
 15  postcode              1834 non-null   str   
 16 

COORDINATES

In [17]:
# Convert latitude and longitude to numeric floats, forcing errors to NaN just in case
master_df_hotels['latitude'] = pd.to_numeric(master_df_hotels['latitude'], errors='coerce')
master_df_hotels['longitude'] = pd.to_numeric(master_df_hotels['longitude'], errors='coerce')

# Verify the fix
print(master_df_hotels[['latitude', 'longitude']].dtypes)

latitude     float64
longitude    float64
dtype: object


STARS AND ROOMS

In [18]:
# Check unique values for stars and rooms
print("--- 'stars' unique values ---")
print(master_df_hotels['stars'].value_counts(dropna=False).head(15))

print("\n--- 'rooms' unique values ---")
print(master_df_hotels['rooms'].value_counts(dropna=False).head(15))

--- 'stars' unique values ---
stars
NaN    2132
3       150
4       144
2        42
3S       19
5        17
4S        6
5S        4
1         2
0         1
Name: count, dtype: int64

--- 'rooms' unique values ---
rooms
NaN    2224
40        6
8         6
6         6
2         6
30        5
4         5
60        5
20        5
37        4
100       4
208       4
162       3
188       3
19        3
Name: count, dtype: int64


In [19]:
# 1. Clean 'rooms' -> convert straight to numeric (missing/malformed become NaN)
master_df_hotels['rooms'] = pd.to_numeric(master_df_hotels['rooms'], errors='coerce')

# 2. Clean 'stars' -> extract the base number (e.g., '3S' becomes 3) 
# and optionally create a flag for 'superior' properties
master_df_hotels['stars_numeric'] = pd.to_numeric(
    master_df_hotels['stars'].astype(str).str.extract(r'(\d+)')[0], 
    errors='coerce'
)

master_df_hotels['is_superior'] = master_df_hotels['stars'].astype(str).str.contains('S', case=False, na=False)

# Check the results
print("Rooms data type:", master_df_hotels['rooms'].dtype)
print("Stars numeric value counts:")
print(master_df_hotels['stars_numeric'].value_counts(dropna=False))
print("\nSuperior hotels found:", master_df_hotels['is_superior'].sum())

Rooms data type: float64
Stars numeric value counts:
stars_numeric
NaN    2132
3.0     169
4.0     150
2.0      42
5.0      21
1.0       2
0.0       1
Name: count, dtype: int64

Superior hotels found: 29


The stars and rooms columns are converted to numeric data types. Non-numeric values are replaced with missing values (NaN) to ensure both columns can be used in further analysis.

### CHECKING DUPLICATES

In [20]:
# Drop duplicates based on OSM ID safely
before_count = len(master_df_hotels)
master_df_hotels = master_df_hotels.drop_duplicates(subset=['@id'])
after_count = len(master_df_hotels)

print(f"Removed {before_count - after_count} duplicate rows.")
print(f"Final unique dataset size: {after_count} accommodations")

Removed 0 duplicate rows.
Final unique dataset size: 2517 accommodations


### CHECKING MISSING VALUES 

Missing values are identified and reviewed across the dataset. Rather than removing all incomplete records, missing information is retained where possible to avoid unnecessary data loss. For overlapping contact fields, available values are combined to fill gaps (already in the loop). Missing city names are filled using the city inferred from the filename (already in the loop). Remaining missing values are kept for further analysis.

In [21]:

# Calculate missing counts and percentages for all columns
missing_df = pd.DataFrame({
    'Missing Count': master_df_hotels.isnull().sum(),
    'Missing (%)': (master_df_hotels.isnull().mean() * 100).round(1)
})

# Filter to show only columns that have at least some missing values, sorted highest first
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

print(f"Total rows in dataset: {len(master_df_hotels)}")
display(missing_df.head(20))

Total rows in dataset: 2517


,Missing Count,Missing (%)
gym,2516,100.0
cabins,2508,99.6
payment:contactless,2508,99.6
caravan_site,2507,99.6
sauna,2504,99.5
swimming_pool,2499,99.3
pets_allowed_clean,2492,99.0
guest_house,2490,98.9
breakfast,2486,98.8
instagram,2486,98.8


Missing pet-policy values are labelled as Unknown rather than assuming that pets are not allowed. This distinguishes accommodations with no recorded information from those with an explicit pet policy and avoids misclassifying missing data. Unknown values are retained for transparency and considered when interpreting the results.

In [22]:
# Fill missing pet tags with 'unknown' so they show up cleanly in filters/groupbys
if 'pets_allowed_clean' in master_df_hotels.columns:
    master_df_hotels['pets_allowed_clean'] = master_df_hotels['pets_allowed_clean'].fillna('unknown')
    
print("Updated 'pets_allowed_clean' distribution:")
print(master_df_hotels['pets_allowed_clean'].value_counts())

Updated 'pets_allowed_clean' distribution:
pets_allowed_clean
unknown    2492
yes          19
no            3
leashed       3
Name: count, dtype: int64


### STANDARDIZING CITY NAMES

City names are standardized by matching different entries to five target cities: Berlin, Hamburg, Leipzig, München, and Hannover. Entries that do not match these cities, including missing or non-string values, are temporarily classified as Other. Hamburg-Neuwerk is explicitly excluded from the Hamburg match. The dataset is then filtered to retain only accommodations in the five target cities. Finally, the dataset shape and number of records per city are checked to review the results.

In [23]:
# Check missing values and distribution for city
print(f"Missing values in 'city': {master_df_hotels['city'].isnull().sum()}")
print("\n--- Breakdown of accommodations per city ---")
print(master_df_hotels['city'].value_counts())

Missing values in 'city': 0

--- Breakdown of accommodations per city ---
city
Berlin                   755
Hamburg                  396
München                  379
Berlin Hotels            266
Leipzig                  179
Hamburg Hotels           145
Munich Hotels            126
Hannover                  96
Leipzig Hotels            88
Hannover Hotels           81
Hamburg-Insel Neuwerk      4
Woltersdorf                2
Name: count, dtype: int64


In [24]:
# cleaining up the city column 

def standardize_city(city_val):
    if not isinstance(city_val, str):
        return "Other"
    c = city_val.lower()
    if "berlin" in c:
        return "Berlin"
    elif "hamburg" in c and "neuwerk" not in c: # Keeping core Hamburg
        return "Hamburg"
    elif "munich" in c or "münchen" in c:
        return "München"
    elif "leipzig" in c:
        return "Leipzig"
    elif "hannover" in c:
        return "Hannover"
    else:
        return "Other"

# Apply standardization
master_df_hotels['city'] = master_df_hotels['city'].apply(standardize_city)

# Filter to keep ONLY the 5 target cities
target_cities = ["Berlin", "Hamburg", "Leipzig", "München", "Hannover"]
master_df_hotels = master_df_hotels[master_df_hotels['city'].isin(target_cities)].copy()

print(f"Dataset shape after filtering to target cities: {master_df_hotels.shape}")
print("\n--- Cleaned City Breakdown ---")
print(master_df_hotels['city'].value_counts())

Dataset shape after filtering to target cities: (2511, 53)

--- Cleaned City Breakdown ---
city
Berlin      1021
Hamburg      541
München      505
Leipzig      267
Hannover     177
Name: count, dtype: int64


### Data Cleaning and Geographic Validation

Empty strings, whitespace-only entries, and common placeholder values are converted to missing values (`NaN`) to ensure consistent handling of incomplete data. Next, latitude and longitude values are checked against Germany's approximate geographic boundaries. Records with missing or out-of-bounds coordinates are identified and excluded from the dataset. Finally, the remaining dataset shape is checked to confirm how many records are available for further spatial analysis.

In [25]:
import numpy as np
import pandas as pd

# 1. Clean up empty strings, whitespace, and placeholder text -> convert to NaN
# Replace empty/whitespace strings across the dataframe
master_df_hotels = master_df_hotels.replace(r'^\s*$', np.nan, regex=True)
# Replace common string placeholders
master_df_hotels = master_df_hotels.replace(['none', 'n/a', 'null', 'nan', 'N/A', 'None'], np.nan)

print("Empty strings cleaned and converted to NaN.")

# 2. Validate coordinates against Germany's geographic bounding box
# Germany bounds roughly: Latitude [47.3, 55.1], Longitude [5.8, 15.0]
lat_valid = master_df_hotels['latitude'].between(47.3, 55.1)
lon_valid = master_df_hotels['longitude'].between(5.8, 15.0)

out_of_bounds = master_df_hotels[~(lat_valid & lon_valid)]
print(f"Rows with missing or out-of-bounds coordinates: {len(out_of_bounds)}")

# Filter to keep only valid, in-bounds records
master_df_hotels = master_df_hotels[lat_valid & lon_valid].copy()

print(f"Final validated dataset shape: {master_df_hotels.shape}")

Empty strings cleaned and converted to NaN.
Rows with missing or out-of-bounds coordinates: 7
Final validated dataset shape: (2504, 53)


# 4. Exporting to CSV

The final cleaned DataFrame is saved as a CSV file using `to_csv()`. The `index=False` parameter prevents Pandas from adding the DataFrame index as an extra column. The exported file can then be loaded into Tableau for visualization and further analysis.

In [26]:
master_df_hotels.to_csv(
    "../data/processed/master_df_hotels.csv",
    index=False,
    encoding="utf-8"
)